# M7 Run 4 - A/B augmentation strength

Attacks the **+0.35 overfit gap** (Run 2). Imbalance fixed to `both` (balanced+focal). Levels: **none / light / rich (=baseline) / strong**. Same fast mini-OOF proxy as Run 3 (train folds 5-8, test folds 1-4 ~58 WPW, 3 seeds @5000, reuse Run 2 cache).

> If `strong` wins -> the overfit gap had room (more reg helps). If `none` >= others -> augmentation is not the lever. Pre-registered bar: beat `rich` beyond its CI, else keep `rich`.

### Block 4.0: Setup and A/B grid

In [ ]:
# M7 Run 4 - A/B AUGMENTATION strength (attacks the +0.35 overfit gap). Imbalance fixed to 'both'.
# Variants: none / light / rich (=Run1-2 baseline) / strong. Same fast mini-OOF proxy as Run 3.
import os, sys, json, time, warnings
import numpy as np, pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score
warnings.filterwarnings('ignore')
ROOT=r'.'
PROCESSED=os.path.join(ROOT,'data','processed'); SRC=os.path.join(ROOT,'src')
METRICS=os.path.join(ROOT,'reports','metrics'); MODELS=os.path.join(ROOT,'models','M7_run4')
CACHE_DIR=os.path.join(PROCESSED,'m7_run2_cache')
for d in (METRICS,MODELS): os.makedirs(d,exist_ok=True)
sys.path.insert(0,SRC)
from evaluation import _boot_ci
RANDOM_STATE=42; RESO=5000
SEEDS=[0,1,2]
EPOCHS=60; PATIENCE=10; BATCH=64; LR=1e-3; WD=1e-4
FOCAL_GAMMA=2.0; FOCAL_ALPHA=0.5; VAL_FRAC=0.15; N_NEG_TRAIN=12000; N_JOBS=6
TEST_FOLDS=[1,2,3,4]; TRAIN_FOLDS=[5,6,7,8]
AUG_LEVELS=['none','light','rich','strong']        # 'rich' = Run 1/2 baseline; A/B axis = strength
BASELINE='rich'
print('M7 Run 4 | A/B augmentation %s (baseline=%s) | both(balanced+focal) | mini-OOF train %s -> test %s @ %d'%(
    AUG_LEVELS,BASELINE,TRAIN_FOLDS,TEST_FOLDS,RESO))
print('Block 4.0 OK.')


### Block 4.1: Reuse Run 2 cache + mini-OOF split

In [ ]:
# Reuse Run 2 cache; mini-OOF split (train folds 5-8, test folds 1-4).
z=np.load(os.path.join(CACHE_DIR,'m7_run2_data.npz'),allow_pickle=True)
X18=z['X18']; fold18=z['fold18']; y18=z['y18']
tr_all=np.where(np.isin(fold18,TRAIN_FOLDS))[0]; te_idx=np.where(np.isin(fold18,TEST_FOLDS))[0]
yte=y18[te_idx]; Xte=X18[te_idx]
print('mini-OOF | train pool %d (%d WPW) | test %d (%d WPW)'%(len(tr_all),int(y18[tr_all].sum()),len(te_idx),int(yte.sum())))
print('Block 4.1 OK.')


### Block 4.2: 1D-ResNet (identical)

In [ ]:
# 1D-ResNet (modest) - identical to Run 1/2/3 (frozen architecture).
import torch, torch.nn as nn
torch.set_num_threads(N_JOBS)
class BasicBlock1d(nn.Module):
    def __init__(self,cin,cout,stride=1):
        super().__init__()
        self.conv1=nn.Conv1d(cin,cout,7,stride=stride,padding=3,bias=False); self.bn1=nn.BatchNorm1d(cout)
        self.conv2=nn.Conv1d(cout,cout,7,padding=3,bias=False); self.bn2=nn.BatchNorm1d(cout)
        self.relu=nn.ReLU(inplace=True); self.down=None
        if stride!=1 or cin!=cout:
            self.down=nn.Sequential(nn.Conv1d(cin,cout,1,stride=stride,bias=False),nn.BatchNorm1d(cout))
    def forward(self,x):
        idt=x if self.down is None else self.down(x)
        o=self.relu(self.bn1(self.conv1(x))); o=self.bn2(self.conv2(o)); return self.relu(o+idt)
class ResNet1d(nn.Module):
    def __init__(self,ch=(16,32,64),pdrop=0.3):
        super().__init__()
        self.stem=nn.Sequential(nn.Conv1d(12,ch[0],15,stride=2,padding=7,bias=False),
                                nn.BatchNorm1d(ch[0]),nn.ReLU(inplace=True),nn.MaxPool1d(4))
        self.layer1=BasicBlock1d(ch[0],ch[0],1); self.layer2=BasicBlock1d(ch[0],ch[1],2); self.layer3=BasicBlock1d(ch[1],ch[2],2)
        self.pool=nn.AdaptiveMaxPool1d(1); self.head=nn.Sequential(nn.Flatten(),nn.Dropout(pdrop),nn.Linear(ch[2],1))
    def forward(self,x):
        return self.head(self.pool(self.layer3(self.layer2(self.layer1(self.stem(x)))))).squeeze(1)
print('Block 4.2 OK | params:', sum(p.numel() for p in ResNet1d().parameters()))


### Block 4.3: Leveled augmentation + training

In [ ]:
# Focal loss + LEVELED augmentation (the A/B axis) + balanced-batch training; early-stop on val focal loss.
def focal_loss(logits,targets,gamma=FOCAL_GAMMA,alpha=FOCAL_ALPHA):
    ce=nn.functional.binary_cross_entropy_with_logits(logits,targets,reduction='none')
    p=torch.sigmoid(logits); pt=torch.where(targets==1,p,1-p)
    a=torch.where(targets==1,torch.full_like(targets,alpha),torch.full_like(targets,1-alpha))
    return (a*(1-pt).pow(gamma)*ce).mean()
def augment(xb,level):
    if level=='none': return xb
    T=xb.shape[2]
    if level=='light':
        sh=max(1,int(0.02*T)); xb=torch.roll(xb,int(torch.randint(-sh,sh+1,(1,)).item()),dims=2)
        return xb+0.01*torch.randn_like(xb)
    P=dict(shift=0.04,scale=0.2,noise=0.02,wander=0.05,ldrop=0.30) if level=='rich' \
      else dict(shift=0.08,scale=0.3,noise=0.03,wander=0.08,ldrop=0.45)     # strong
    sh=max(1,int(P['shift']*T)); xb=torch.roll(xb,int(torch.randint(-sh,sh+1,(1,)).item()),dims=2)
    xb=xb*((1.0-P['scale'])+2*P['scale']*torch.rand(xb.shape[0],1,1))
    xb=xb+P['noise']*torch.randn_like(xb)
    t=torch.linspace(0,1,T).view(1,1,-1); fb=0.5+2.0*torch.rand(xb.shape[0],1,1); ph=2*np.pi*torch.rand(xb.shape[0],1,1)
    xb=xb+P['wander']*torch.sin(2*np.pi*fb*t+ph)
    if torch.rand(1).item()<P['ldrop']: xb[:,int(torch.randint(0,12,(1,)).item()),:]=0.0
    if level=='strong' and torch.rand(1).item()<0.25: xb[:,int(torch.randint(0,12,(1,)).item()),:]=0.0
    return xb
def predict(model,X):
    model.eval(); out=[]
    with torch.no_grad():
        for i in range(0,len(X),256):
            out.append(torch.sigmoid(model(torch.tensor(np.ascontiguousarray(X[i:i+256],dtype=np.float32)))).numpy())
    return np.nan_to_num(np.concatenate(out))
def train_one(seed,Xtr,ytr,Xva,yva,level):
    torch.manual_seed(seed); np.random.seed(seed); rng=np.random.default_rng(seed)
    pos=np.where(ytr==1)[0]; neg=np.where(ytr==0)[0]; half=BATCH//2
    Xv=torch.tensor(np.ascontiguousarray(Xva,dtype=np.float32)); yv=torch.tensor(yva)
    model=ResNet1d(); opt=torch.optim.Adam(model.parameters(),lr=LR,weight_decay=WD)
    steps=max(1,len(ytr)//BATCH); best=1e9; best_state=None; bad=0
    for ep in range(EPOCHS):
        model.train()
        for _ in range(steps):
            bi=np.concatenate([rng.choice(pos,half,True),rng.choice(neg,half,True)])
            xb=augment(torch.tensor(np.ascontiguousarray(Xtr[bi],dtype=np.float32)),level); yb=torch.tensor(ytr[bi])
            opt.zero_grad(); loss=focal_loss(model(xb),yb); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),5.0); opt.step()
        model.eval()
        with torch.no_grad():
            vl=[focal_loss(model(Xv[i:i+256]),yv[i:i+256]).item() for i in range(0,len(Xv),256)]
        vloss=float(np.mean(vl))
        if vloss<best-1e-4: best=vloss; best_state={k:v.clone() for k,v in model.state_dict().items()}; bad=0
        else: bad+=1
        if bad>=PATIENCE: break
    model.load_state_dict(best_state); return model
print('Block 4.3 OK.')


### Block 4.4: A/B loop (checkpointed per level+seed)

In [ ]:
# A/B loop over augmentation levels. Fixed train pool/val split (fair A/B). Checkpoint per (level, seed).
def build_pool(seed0):
    rng=np.random.default_rng(seed0)
    pos=tr_all[y18[tr_all]==1]; negall=tr_all[y18[tr_all]==0]
    neg=rng.choice(negall,min(N_NEG_TRAIN,len(negall)),replace=False)
    vp=pos.copy(); vn=neg.copy(); rng.shuffle(vp); rng.shuffle(vn)
    nvp=max(6,int(VAL_FRAC*len(vp))); nvn=int(VAL_FRAC*len(vn))
    val=np.concatenate([vp[:nvp],vn[:nvn]]); pool=np.concatenate([pos,neg]); tr=np.setdiff1d(pool,val)
    return tr,val
tr_idx,val_idx=build_pool(2024)
Xtr=np.ascontiguousarray(X18[tr_idx],dtype=np.float32); ytr=y18[tr_idx]; Xva=X18[val_idx]; yva=y18[val_idx]
print('pool: train %d (%d WPW) | val %d (%d WPW)'%(len(tr_idx),int(ytr.sum()),len(val_idx),int(yva.sum())))
CK=os.path.join(MODELS,'run4_ab_ckpt.npz'); store={}
if os.path.exists(CK):
    store=np.load(CK,allow_pickle=True)['store'].item(); print('[ckpt] resumed:',list(store.keys()))
rows=[]; t0=time.time()
for level in AUG_LEVELS:
    seed_pred=[np.asarray(p) for p in store.get(level,{}).get('pred',[])]; done=store.get(level,{}).get('done',0)
    for si,s in enumerate(SEEDS):
        if si<done: continue
        model=train_one(s,Xtr,ytr,Xva,yva,level)
        seed_pred.append(predict(model,Xte))
        store[level]={'pred':[p.tolist() for p in seed_pred],'done':si+1}; np.savez(CK,store=np.array(store,dtype=object))
    ens=np.mean(seed_pred,0); ap=float(average_precision_score(yte,ens)); auc=float(roc_auc_score(yte,ens))
    lo,hi=_boot_ci(yte.astype(int),ens,average_precision_score); seed_ap=[float(average_precision_score(yte,p)) for p in seed_pred]
    rows.append(dict(level=level,AP=ap,AP_lo=lo,AP_hi=hi,AUC=auc,AP_seed_mean=float(np.mean(seed_ap)),AP_seed_std=float(np.std(seed_ap))))
    print('  [%-6s] AP %.3f CI[%.3f,%.3f] | per-seed %.3f+/-%.3f | %.1f min'%(level,ap,lo,hi,np.mean(seed_ap),np.std(seed_ap),(time.time()-t0)/60))
ab=pd.DataFrame(rows)
print('Block 4.4 OK.'); print(ab.to_string(index=False))


### Block 4.5: Ranking + decision

In [ ]:
# Ranking + decision. Pre-registered bar: beat baseline ('rich') beyond its CI; else keep 'rich'.
base=ab[ab.level==BASELINE].iloc[0]; ab_sorted=ab.sort_values('AP',ascending=False).reset_index(drop=True); winner=ab_sorted.iloc[0]
beats=bool(winner.AP_lo>base.AP and winner.level!=BASELINE); chosen=winner.level if beats else BASELINE
print('=== M7 Run 4 - augmentation A/B (mini-OOF, test folds 1-4) ===')
print(ab_sorted.to_string(index=False))
print('  baseline %s: AP %.3f CI[%.3f,%.3f]'%(BASELINE,base.AP,base.AP_lo,base.AP_hi))
print('  top: %s AP %.3f CI[%.3f,%.3f] | beats beyond CI ? %s -> ADOPT: %s'%(winner.level,winner.AP,winner.AP_lo,winner.AP_hi,beats,chosen))
print('  NOTE: ranking proxy (4-fold train). If "strong" wins, the +0.35 overfit gap had room; if "none">=others, aug is not the lever.')
json.dump({'levels':ab_sorted.to_dict('records'),'baseline':BASELINE,'baseline_AP':float(base.AP),
           'winner':winner.level,'beats_beyond_CI':beats,'adopted':chosen},
          open(os.path.join(METRICS,'m7_run4_augmentation_ab.json'),'w'),indent=2,default=float)
print('Block 4.5 OK | adopted augmentation: %s'%chosen)
